# Sizing a myLedger node

**What this answers:** how much memory and how much disk, for a stated demand and a stated policy.

**What it does not answer:** throughput and tail latency. Those are not arithmetic — design notes
§10 records a hardware profile that was built, measured and *refused*, because the stages' cache
misses overlap and the formula priced each at full latency. Ask `ledgersim` instead, which runs the
real reactor on a virtual clock.

## The split

| | owns | why |
|---|---|---|
| the code | what one unit costs | it is `size_of`, known at build time |
| this notebook | how many units | it follows from a rate, a lifetime and a retention |

Nothing here hard-codes a byte count. They come from `sizing/units.json`, which is
`ledgerfio layout --json` with the commit it was taken at. Refresh it with `make sizing-units`
after changing a sized struct; `make verify` fails if it is stale.


In [ ]:
import sys; sys.path.insert(0, '.')
from model import (Demand, Policy, Dials, Sizing, report,
                   buckets_for, index_slots_for, gigabytes, check_bucket_rule, load_units)

units = load_units()
check_bucket_rule(units)   # the one piece of the code's arithmetic reproduced here
print(units['source'], 'at', units['commit'][:12])
print(len(units['parts']), 'sized structures')


## The inputs

**Three rates, not one.** A peak decides what is held in flight, the busiest hour decides the
windows an hour wide, and the day decides retention. A deployment whose peak is eighty-six times
its mean is sized eighty-six times wrong by whichever single number it picks.

**Dials are outputs, not inputs.** `Dials` is here so a plan can be checked against the ceilings a
node is configured with — a dial below what demand requires is where the node refuses work.


In [ ]:
demand = Demand(
    peak_rate=300_000,          # tx/s at the peak
    peak_seconds=60,            # how long that peak is sustained -- rate alone decides nothing
    busiest_hour_tx=30_000_000, # the hour-wide windows follow this
    daily_tx=300_000_000,       # retention and disk follow this
    accounts=10_000_000,
    records_per_tx=1.0,         # kind mix: pending records one tx creates
    hold_share=1.0,             # share of tx that create a hold
    short_life_seconds=1.0,     # how long a hold that resolves normally lives
    survivor_share=0.5,         # share still unresolved when retention ends
    commit_latency_seconds=0.010,
)

policy = Policy(
    retention_days=2, grace_days=1,
    flush_window_hours=1,       # a recovery bound: what a restart replays
    residency_hours=24,         # a latency bound: how far back memory answers
    idem_window_hours=1,        # duplicate detection -- not enforced by the code yet
    snapshot_every_effects=1_000_000,
)

print(demand.sanity() or 'inputs are consistent')


In [ ]:
sizing = Sizing(demand, policy, Dials())
print(report(sizing))


## Changing a number

Any unit cost can be overridden to ask *what if this struct were smaller*. **The report says so.**
A hypothetical printed beside measurements reads as a measurement, which is the same rule every
benchmark here follows when it prints its thread placement.


In [ ]:
what_if = Sizing(demand, policy, Dials(), overrides={'pending index': 4})
print(report(what_if).splitlines()[0])
print()
print(f"measured  {gigabytes(sizing.memory_bytes):.2f} GB")
print(f"overridden {gigabytes(what_if.memory_bytes):.2f} GB")


## The staircase

A hash table rounds its bucket count to a power of two, so **one percent more entries can double
the memory**. The cuckoo index steps too, four times more coarsely. A single point tells you
nothing about which side of a step it is on — sweep.


In [ ]:
print(f"{'daily tx':>14}{'live holds':>16}{'index slots':>16}{'index GB':>11}")
for daily in (100_000_000, 200_000_000, 300_000_000, 400_000_000, 600_000_000, 800_000_000):
    one = Sizing(Demand(300_000, 60, min(daily, 30_000_000), daily, 10_000_000), policy)
    line = dict(one.lines_by_name)['pending index']
    print(f'{daily:>14,}{one.live_holds:>16,}{line.count:>16,}{gigabytes(line.bytes):>11.2f}')


### The idem map, and why the peak's *duration* is an input

The idem window is an hour, so its count is the busiest hour's transactions — there is no queue to
absorb a peak into. This is the structure where the peak-to-mean ratio decides everything.


In [ ]:
print(f"{'busiest hour tx':>18}{'buckets':>16}{'idem GB':>10}")
for hour in (3_000_000, 10_000_000, 30_000_000, 100_000_000, 300_000_000, 1_080_000_000):
    one = Sizing(Demand(300_000, 60, hour, max(hour, 300_000_000), 10_000_000), policy)
    line = dict(one.lines_by_name)['idem keys']
    print(f'{hour:>18,}{line.count:>16,}{gigabytes(line.bytes):>10.2f}')
print()
print('a peak of 300k/s sustained for a whole hour is the last row: 1.08G keys.')


## What is not sized here

- **Throughput and the tail** — `ledgersim capacity` and `ledgersim require`, which run the real
  reactor rather than a formula.
- **`skew`** — hot-account concentration costs lane contention, which is latency, not bytes.
- **The idem window** — one hour is the intended window and the count above assumes it, but the
  rotating generations that would enforce it are not built. Today the map only grows.
- **`kept log`** — there is no compaction, so its count is a snapshot cadence rather than a steady
  state.
